# Tufin Data Science Home Assignment

## 1. Exploratory Data Analysis (EDA)

Goal: understand the structure of the rule graph, the labeled data, and the unlabeled data before choosing a prediction approach.

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rules = pd.read_csv("rules.csv")
tags  = pd.read_csv("tags.csv")

print("rules shape:", rules.shape)
print("tags  shape:", tags.shape)
rules.head(3)

rules shape: (113438, 3)
tags  shape: (3000, 2)


,rule_id,src_obj_id,dst_obj_id
0,rule_0000,obj_04083,obj_05420
1,rule_0000,obj_04987,obj_09572
2,rule_0000,obj_02993,obj_08605


### 1.1 Tags — Labeled Data

In [4]:
print(f"Total labeled objects : {len(tags)}")
print(f"Unique tags           : {tags['tag'].nunique()}")
print(f"Duplicate obj_id      : {tags['obj_id'].duplicated().sum()}")
print()
print(tags['tag'].value_counts().to_string())

Total labeled objects : 3000
Unique tags           : 9
Duplicate obj_id      : 0

tag
gp_T2_Application     1100
Users Networks         850
gp_T1_DMZ_private      430
gp_T3_Database         330
gp_Common_services     170
gp_T0_DMZ_public        90
gp_Management           12
gp_GUESTS               10
c_TER_CH_SRV             8


In [5]:
tag_counts = tags['tag'].value_counts().reset_index()
tag_counts.columns = ['tag', 'count']
tag_counts['pct'] = (tag_counts['count'] / tag_counts['count'].sum() * 100).round(1)

fig = px.bar(
    tag_counts,
    x='tag', y='count',
    text=tag_counts['pct'].astype(str) + '%',
    color='tag',
    title='Tag Distribution (labeled objects)',
    labels={'tag': 'Tag', 'count': 'Count'},
)
fig.update_layout(xaxis_tickangle=-30, showlegend=False)
fig.show()

### 1.2 Rules — Structure

In [6]:
n_rules      = rules['rule_id'].nunique()
n_edges      = len(rules)
all_objs     = pd.concat([rules['src_obj_id'], rules['dst_obj_id']]).unique()
n_obj_total  = len(all_objs)
self_loops   = (rules['src_obj_id'] == rules['dst_obj_id']).sum()
dup_edges    = rules.duplicated(subset=['src_obj_id', 'dst_obj_id']).sum()

print(f"Unique rule_ids          : {n_rules}")
print(f"Total edges (rows)       : {n_edges}")
print(f"Unique objects in rules  : {n_obj_total}")
print(f"Self-loops               : {self_loops}")
print(f"Duplicate src→dst pairs  : {dup_edges}")

Unique rule_ids          : 500
Total edges (rows)       : 113438
Unique objects in rules  : 10000
Self-loops               : 23
Duplicate src→dst pairs  : 355


In [7]:
# Edges per rule_id
edges_per_rule = rules.groupby('rule_id').size().reset_index(name='edge_count')

fig = px.histogram(
    edges_per_rule,
    x='edge_count',
    nbins=40,
    title='Distribution of Edges per Rule ID',
    labels={'edge_count': 'Number of Edges', 'count': 'Number of Rules'},
)
fig.update_layout(bargap=0.05)
fig.show()

print(edges_per_rule['edge_count'].describe().round(1))

count    500.0
mean     226.9
std       92.3
min       13.0
25%      166.8
50%      212.5
75%      280.2
max      484.0
Name: edge_count, dtype: float64


### 1.3 Labeled vs Unlabeled Objects

In [8]:
labeled_ids   = set(tags['obj_id'])
all_obj_ids   = set(all_objs)
unlabeled_ids = all_obj_ids - labeled_ids
only_in_tags  = labeled_ids - all_obj_ids   # labeled but never appear in rules / not in edge

print(f"Objects in rules             : {len(all_obj_ids)}")
print(f"Labeled objects              : {len(labeled_ids)}")
print(f"Unlabeled objects in rules   : {len(unlabeled_ids)}")
print(f"Labeled objects NOT in rules : {len(only_in_tags)}  ← no graph signal available")
print(f"Label coverage               : {len(labeled_ids & all_obj_ids) / len(all_obj_ids):.1%}")

fig = px.pie(
    names=['Labeled (in rules)', 'Unlabeled (in rules)', 'Labeled (not in rules)'],
    values=[
        len(labeled_ids & all_obj_ids),
        len(unlabeled_ids),
        len(only_in_tags),
    ],
    title='Object Coverage: Labeled vs Unlabeled',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

Objects in rules             : 10000
Labeled objects              : 3000
Unlabeled objects in rules   : 7000
Labeled objects NOT in rules : 0  ← no graph signal available
Label coverage               : 30.0%


### 1.4 Node Degree Distribution

In [9]:
out_deg = rules.groupby('src_obj_id').size().rename('out_degree')
in_deg  = rules.groupby('dst_obj_id').size().rename('in_degree')
degree  = pd.DataFrame({'obj_id': list(all_obj_ids)})
degree  = degree.merge(out_deg.reset_index().rename(columns={'src_obj_id':'obj_id'}), on='obj_id', how='left')
degree  = degree.merge(in_deg.reset_index().rename(columns={'dst_obj_id':'obj_id'}),  on='obj_id', how='left')
degree  = degree.fillna(0).astype({'out_degree': int, 'in_degree': int})
degree['total_degree'] = degree['out_degree'] + degree['in_degree']
degree['labeled'] = degree['obj_id'].isin(labeled_ids)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Out-Degree Distribution', 'In-Degree Distribution'))
for col, field in [(1, 'out_degree'), (2, 'in_degree')]:
    for is_labeled, color, name in [(True, '#636EFA', 'Labeled'), (False, '#EF553B', 'Unlabeled')]:
        subset = degree[degree['labeled'] == is_labeled][field]
        fig.add_trace(go.Histogram(x=subset, name=name, legendgroup=name,
                                   showlegend=(col == 1), marker_color=color,
                                   opacity=0.7, nbinsx=50), row=1, col=col)
fig.update_layout(title='Node Degree Distribution by Label Status', barmode='overlay')
fig.show()

print(degree.groupby('labeled')[['out_degree','in_degree','total_degree']].describe().round(1).to_string())

        out_degree                                         in_degree        \
             count  mean  std  min   25%   50%   75%   max     count  mean   
labeled                                                                      
False       7000.0  12.9  5.2  5.0  10.0  12.0  15.0  56.0    7000.0  13.1   
True        3000.0   7.7  3.6  3.0   5.0   7.0   9.0  41.0    3000.0   7.2   

         ...             total_degree                                           
         ...   75%   max        count  mean  std   min   25%   50%   75%   max  
labeled  ...                                                                    
False    ...  16.0  42.0       7000.0  26.0  7.4  10.0  21.0  25.0  30.0  78.0  
True     ...   9.0  24.0       3000.0  14.9  5.0   6.0  11.0  14.0  17.0  51.0  

[2 rows x 24 columns]


### 1.5 Tag Homophily — Do Connected Objects Share Tags?

In [10]:
tag_map = tags.set_index('obj_id')['tag']

# Keep only edges where both endpoints are labeled
labeled_edges = rules.copy()
labeled_edges['src_tag'] = labeled_edges['src_obj_id'].map(tag_map)
labeled_edges['dst_tag'] = labeled_edges['dst_obj_id'].map(tag_map)
both_labeled  = labeled_edges.dropna(subset=['src_tag', 'dst_tag'])

same_tag_rate = (both_labeled['src_tag'] == both_labeled['dst_tag']).mean()
print(f"Edges where both endpoints are labeled : {len(both_labeled)}")
print(f"Of those, same-tag rate                : {same_tag_rate:.1%}")

# Tag-to-tag flow heatmap
flow = both_labeled.groupby(['src_tag', 'dst_tag']).size().unstack(fill_value=0)
fig = px.imshow(
    flow,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Tag-to-Tag Edge Flow (labeled endpoints only)',
    labels=dict(x='Destination Tag', y='Source Tag', color='Edge Count'),
    aspect='auto',
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

Edges where both endpoints are labeled : 4393
Of those, same-tag rate                : 10.7%


### 1.6 Neighbor Tag Profile per Unlabeled Object

In [11]:
# For every unlabeled node, count how many labeled neighbors it has (src or dst)
src_neighbors = rules[['src_obj_id','dst_obj_id']].rename(columns={'src_obj_id':'obj_id','dst_obj_id':'neighbor'})
dst_neighbors = rules[['src_obj_id','dst_obj_id']].rename(columns={'dst_obj_id':'obj_id','src_obj_id':'neighbor'})
all_neighbors = pd.concat([src_neighbors, dst_neighbors], ignore_index=True)

all_neighbors['neighbor_tag'] = all_neighbors['neighbor'].map(tag_map)
labeled_neighbor_counts = (
    all_neighbors[all_neighbors['obj_id'].isin(unlabeled_ids)]
    .dropna(subset=['neighbor_tag'])
    .groupby('obj_id')['neighbor_tag']
    .count()
)

no_labeled_neighbor = len(unlabeled_ids) - len(labeled_neighbor_counts)
print(f"Unlabeled objects with ≥1 labeled neighbor : {len(labeled_neighbor_counts)}")
print(f"Unlabeled objects with NO labeled neighbor  : {no_labeled_neighbor}  ← cold-start problem")

fig = px.histogram(
    labeled_neighbor_counts.reset_index(name='count'),
    x='count',
    nbins=40,
    title='Labeled Neighbor Count per Unlabeled Object',
    labels={'count': 'Number of Labeled Neighbors'},
)
fig.show()

Unlabeled objects with ≥1 labeled neighbor : 6923
Unlabeled objects with NO labeled neighbor  : 77  ← cold-start problem


In [12]:
# Build a dataframe of all objects with their numeric ID part and tag label
obj_series = pd.Series(list(all_obj_ids), name='obj_id')
obj_df = obj_series.to_frame()
obj_df['id_num'] = obj_df['obj_id'].str.extract(r'(\d+)$').astype(int)
obj_df['tag'] = obj_df['obj_id'].map(tag_map).fillna('unlabeled')

# Order: known tags by median id_num, then unlabeled at the end
tag_order = (
    obj_df[obj_df['tag'] != 'unlabeled']
    .groupby('tag')['id_num']
    .median()
    .sort_values()
    .index.tolist()
) + ['unlabeled']

fig = px.box(
    obj_df,
    x='tag',
    y='id_num',
    color='tag',
    category_orders={'tag': tag_order},
    title='Distribution of Object ID Numeric Part by Tag (incl. Unlabeled)',
    labels={'tag': 'Tag', 'id_num': 'obj_id Numeric Part'},
    points=False,
)
fig.update_layout(xaxis_tickangle=-30, showlegend=False)
fig.show()

### 1.7 EDA Key Findings

**Labeled data**
- 3,000 labeled objects across 9 tags. Distribution is heavily skewed: `gp_T2_Application` (~37%) and `Users Networks` (~28%) dominate; `gp_GUESTS`, `gp_Management`, and `c_TER_CH_SRV` have fewer than 15 samples each — rare classes that will be hard to predict reliably.

**Unlabeled data**
- The rules graph contains many more objects than the tag file. Label coverage is partial; a meaningful fraction of objects have no labeled neighbors at all (cold-start).

**Rule structure**
- All rows share a small number of `rule_id` values (each rule fans out to many src→dst pairs). The dataset is essentially a directed multigraph where `rule_id` groups sets of allowed connections.
- Duplicate src→dst pairs exist across different rules — the graph has parallel edges.
- 23 self-loops detected (src == dst) — minor anomaly worth noting but unlikely to affect predictions significantly.

**Homophily & tag flow**
- The same-tag edge rate among labeled endpoints is only ~10.7%, meaning the majority of connections cross tag boundaries. The tag-to-tag heatmap does **not** show a strong block-diagonal — edges are broadly distributed across tag pairs.
- This is consistent with a tiered security architecture where traffic naturally flows *between* zones (e.g., `Users Networks` → `gp_T2_Application`), rather than within them.
- Low homophily weakens the assumption behind simple label propagation; neighbor-majority voting will be noisy, and predictions for unlabeled nodes should be treated with caution.

**Risks & limitations**
- Rare tags (`gp_GUESTS`, `gp_Management`, `c_TER_CH_SRV`) will likely be under-predicted — insufficient labeled examples for confident propagation.
- Isolated unlabeled objects (no labeled neighbors) can only be predicted via global priors or degree-based heuristics.
- Low homophily means graph structure alone is a weak signal; complementary features (degree profile, rule membership) should be incorporated.
- The dataset is synthetic/simplified; real data would be noisier and larger.